In [16]:
import re
import time

import numpy as np
import requests
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
ai = OpenAI()

ENC = tiktoken.get_encoding("cl100k_base")
ntok = lambda t: len(ENC.encode(t))

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-mini"

In [17]:
TITLES = ["정보 보안", "피싱", "SQL 삽입", "크로스 사이트 스크립팅"]

session = requests.Session()
session.headers.update({"User-Agent": "llm-edu-notebook/1.0"})   # 한글 금지 (latin-1만 허용)


def fetch_wiki(title, retries=4):
    """위키백과에서 문서 한 개를 순수 텍스트로 받아온다. 429가 오면 기다렸다 재시도."""
    for attempt in range(retries):
        resp = session.get(
            "https://ko.wikipedia.org/w/api.php",
            params={
                "action": "query", "format": "json", "prop": "extracts",
                "explaintext": 1, "redirects": 1, "titles": title,
            },
            timeout=30,
        )
        if resp.status_code == 429:          # 요청이 너무 잦음 -> 기다렸다 재시도
            wait = 3 * (attempt + 1)
            print(f"  429 Too Many Requests — {wait}초 대기 후 재시도")
            time.sleep(wait)
            continue
        page = next(iter(resp.json()["query"]["pages"].values()))
        return page["title"], page["extract"].strip()
    raise RuntimeError(f"{title} 내려받기 실패")


docs = {}
for t in TITLES:
    title, text = fetch_wiki(t)
    docs[title] = text
    print(f"{title:14s} {len(text):>7,}자 {ntok(text):>7,}토큰")
    time.sleep(1.5)                          # 위키백과에 대한 예의이자 429 예방

print(f"\n합계 {sum(len(t) for t in docs.values()):,}자 "
      f"{sum(ntok(t) for t in docs.values()):,}토큰 ({len(docs)}개 문서)")

정보 보안            4,065자   3,462토큰
피싱               5,120자   4,363토큰
SQL 삽입           2,397자   1,438토큰
사이트 간 스크립팅       2,324자   1,817토큰

합계 13,906자 11,080토큰 (4개 문서)


In [18]:
def chunk_fixed(text, size=200, overlap=0):
    """`size` 토큰마다 무조건 자른다. 문장이 어디서 끝나는지는 신경 쓰지 않는다."""
    ids = ENC.encode(text)
    step = size - overlap
    out = []
    for i in range(0, len(ids), step):
        piece = ids[i : i + size]
        if not piece:
            break
        out.append(ENC.decode(piece))
        if i + size >= len(ids):
            break
    return out


def chunk_paragraphs(text, target=200):
    """문단을 통째로 모아서 대략 `target` 토큰이 될 때까지 합친다."""
    paras = [p.strip() for p in re.split(r"\n+", text) if p.strip()]
    out, buf, n = [], [], 0
    for p in paras:
        t = ntok(p)
        if buf and n + t > target:
            out.append("\n\n".join(buf))
            buf, n = [], 0
        buf.append(p)
        n += t
    if buf:
        out.append("\n\n".join(buf))
    return out


def chunk_all(docs, chunker, **kw):
    """문서마다 따로 청킹한 뒤 합친다. 각 청크에 출처를 붙인다."""
    items = []
    for title, text in docs.items():
        for piece in chunker(text, **kw):
            items.append({"text": piece, "source": title, "tokens": ntok(piece)})
    return items

In [19]:
CHUNKINGS = {
    "fixed-200": chunk_all(docs, chunk_fixed, size=200),
    "paragraphs-200": chunk_all(docs, chunk_paragraphs, target=200),
}

for name, items in CHUNKINGS.items():
    sizes = [it["tokens"] for it in items]
    per_doc = {t: sum(1 for it in items if it["source"] == t) for t in docs}
    print(f"{name:16s} {len(items):>3}개 청크  중앙값 {int(np.median(sizes)):>3}토큰  "
          f"최대 {max(sizes):>4}토큰")
    print(f"{'':16s} 문서별: {per_doc}")

fixed-200         58개 청크  중앙값 200토큰  최대  201토큰
                 문서별: {'정보 보안': 18, '피싱': 22, 'SQL 삽입': 8, '사이트 간 스크립팅': 10}
paragraphs-200    64개 청크  중앙값 170토큰  최대  536토큰
                 문서별: {'정보 보안': 22, '피싱': 25, 'SQL 삽입': 7, '사이트 간 스크립팅': 10}


In [20]:
def embed(texts, batch=256):
    out = []
    for i in range(0, len(texts), batch):
        r = ai.embeddings.create(model=EMBED_MODEL, input=texts[i : i + batch])
        out.extend(d.embedding for d in r.data)
    V = np.asarray(out, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True)   # 단위 벡터 -> 내적 == 코사인 유사도
    return V

In [21]:
import chromadb
from chromadb.errors import NotFoundError

DB_PATH = "chroma_chunk_ko3"
client = chromadb.PersistentClient(path=DB_PATH)

collections = {}
for name, items in CHUNKINGS.items():
    texts = [it["text"] for it in items]
    V = embed(texts)
    print(f"{name:16s} 임베딩 완료 (약 ${sum(it['tokens'] for it in items) / 1e6 * 0.02:.4f})")

    # 이미 있으면 지우고 새로 만든다. (셀을 몇 번이든 다시 실행할 수 있도록)
    try:
        client.delete_collection(name)
    except NotFoundError:
        pass

    col = client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    col.add(
        ids=[f"{name}-{i}" for i in range(len(items))],
        documents=texts,
        embeddings=V.tolist(),
        metadatas=[{"source": it["source"], "chunk_id": i, "tokens": it["tokens"]}
                   for i, it in enumerate(items)],
    )
    collections[name] = col
    print(f"{name:16s} {col.count():>3}개 청크 저장")

print(f"\n저장 위치: {DB_PATH}/")

fixed-200        임베딩 완료 (약 $0.0002)
fixed-200         58개 청크 저장
paragraphs-200   임베딩 완료 (약 $0.0002)
paragraphs-200    64개 청크 저장

저장 위치: chroma_chunk_ko3/


In [22]:
def search_db(query, name, k=3):
    """(점수, 본문, 출처, 토큰수) 리스트를 돌려준다."""
    qv = embed([query])[0]
    res = collections[name].query(query_embeddings=[qv.tolist()], n_results=k)
    return [
        {"score": 1 - dist, "text": text, "source": meta["source"], "tokens": meta["tokens"]}
        for dist, text, meta in zip(res["distances"][0], res["documents"][0], res["metadatas"][0])
    ]


for q in ["피싱 메일을 어떻게 구별하나요?", "SQL 인젝션을 막는 방법은?"]:
    print(f"Q: {q}")
    for hit in search_db(q, "paragraphs-200", k=3):
        print(f"  [{hit['score']:.3f}] ({hit['source']}) {' '.join(hit['text'].split())[:70]}...")
    print()

Q: 피싱 메일을 어떻게 구별하나요?
  [0.601] (피싱) 웹 사이트를 만드는 기술 이외의 특별한 기술은 아무것도 필요가 없다. 피싱 사기를 하는 방법으로 웹사이트를 만들고 메일을 보낸...
  [0.534] (피싱) 백신 소프트웨어에 검출되지 않는 경우가 있다. 피싱 사기의 경우 아무런 특색 없는 단순한 메일 형태로 첨부파일 등이 없는 HT...
  [0.492] (피싱) 메일을 이용해서 신뢰할 수 있는 메일 주소로 가장한다. 피싱 메일은 대부분 송신자를 사칭하고 있다. 예를 들어 사기꾼이 시티은...

Q: SQL 인젝션을 막는 방법은?
  [0.458] (SQL 삽입) SQL 삽입(영어: SQL Injection, SQL 인젝션, SQL 주입)은 응용 프로그램 보안 상의 허점을 의도적으로 이용...
  [0.432] (SQL 삽입) Blind SQL 삽입은 평범한 SQL 삽입과 같이 원하는 데이터를 가져올 쿼리를 삽입하는 기술이다. 이것은 웹에서 SQL 삽...
  [0.404] (SQL 삽입) DBMS가 쿼리문을 분석하고 최적화를 진행한다. 실행: 플레이스홀더에 입력할 사용자 입력 값이 전송되고 DBMS가 쿼리문을 실...



In [23]:
SYSTEM_PROMPT = """당신은 정보보안 교육 자료를 안내하는 도우미입니다.

아래 [자료]에 있는 내용만 근거로 답변하세요.
- [자료]에 없는 내용은 추측하지 말고 "제공된 자료에서 찾을 수 없습니다"라고 답하세요.
- 문장 끝에 근거가 된 자료 번호를 [1], [2] 형식으로 표시하세요.
- 한국어로, 3~5문장으로 간결하게 답하세요.

[자료]
{context}
"""


def build_context(hits):
    """검색 결과를 번호와 출처가 붙은 컨텍스트 문자열로 만든다."""
    blocks = []
    for i, hit in enumerate(hits, 1):
        blocks.append(f"[{i}] (출처: {hit['source']})\n{hit['text']}")
    return "\n\n".join(blocks)


def answer(query, name="paragraphs-200", k=3):
    """검색 -> 컨텍스트 구성 -> LLM 답변 생성."""
    hits = search_db(query, name, k)
    response = ai.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT.format(context=build_context(hits))},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content, hits

In [24]:
text, hits = answer("피싱 공격은 어떤 방식으로 이루어지나요?")
print(text)
print("\n--- 근거로 사용한 청크 ---")
for i, hit in enumerate(hits, 1):
    print(f"  [{i}] ({hit['source']}) 점수 {hit['score']:.3f} · {hit['tokens']}토큰")

피싱 공격은 전자우편이나 메신저를 이용해 신뢰할 수 있는 사람이나 기업이 보낸 것처럼 가장하여 비밀번호, 신용카드 정보 등 기밀 정보를 부정하게 얻으려는 소셜 엔지니어링 기법입니다. 사용자를 속여 금융 정보나 패스워드를 '낚는' 방식으로 이루어집니다. 최근에는 컴퓨터뿐 아니라 전화를 이용한 보이스피싱도 등장하는 등 다양한 형태가 있습니다[1][3].

--- 근거로 사용한 청크 ---
  [1] (피싱) 점수 0.453 · 181토큰
  [2] (피싱) 점수 0.400 · 148토큰
  [3] (피싱) 점수 0.397 · 264토큰


In [25]:
text, hits = answer("랜섬웨어에 감염되면 몇 분 안에 신고해야 하나요?")
print(text)
print("\n--- 검색된 청크 (점수가 낮습니다) ---")
for hit in hits:
    print(f"  [{hit['score']:.3f}] ({hit['source']})")

제공된 자료에서 랜섬웨어 감염 시 몇 분 안에 신고해야 한다는 구체적인 시간 정보는 찾을 수 없습니다[3]. 다만, 피싱 의심 메일을 받았을 경우에는 신속히 해당 기관이나 한국정보보호진흥원에 신고하는 것이 권장됩니다[2].

--- 검색된 청크 (점수가 낮습니다) ---
  [0.368] (피싱)
  [0.343] (피싱)
  [0.338] (정보 보안)


In [26]:
QUESTIONS = [
    "정보 보안의 세 가지 핵심 요소는 무엇인가요?",   # 답이 한 곳에 명확히 있음
    "피싱 메일을 어떻게 구별하나요?",               # 답이 여러 곳에 흩어져 있음
]

for question in QUESTIONS:
    print("#" * 72)
    print(f"# {question}")
    print("#" * 72)
    for name in ["fixed-200", "paragraphs-200"]:
        text, hits = answer(question, name, k=3)
        print(f"\n[{name}]  1위 점수 {hits[0]['score']:.3f}  "
              f"출처: {', '.join(h['source'] for h in hits)}")
        print(text)
    print()

########################################################################
# 정보 보안의 세 가지 핵심 요소는 무엇인가요?
########################################################################

[fixed-200]  1위 점수 0.561  출처: 정보 보안, 정보 보안, 정보 보안
정보 보안의 세 가지 핵심 요소는 기밀성(Confidentiality), 무결성(Integrity), 가용성(Availability)입니다. 이 요소들은 정보가 허가된 사람만 접근 가능하고, 정보가 변조되지 않으며, 필요할 때 언제든지 사용할 수 있도록 보장하는 것을 의미합니다[3].

[paragraphs-200]  1위 점수 0.556  출처: 정보 보안, 정보 보안, 정보 보안
정보 보안의 세 가지 핵심 요소는 기밀성(Confidentiality), 무결성(Integrity), 가용성(Availability)입니다. 이는 인가된 사용자만이 정확하고 완전한 정보에 필요할 때 접근할 수 있도록 하는 것을 의미합니다. 추가적으로 진정성, 책임성, 부인방지, 신뢰성과도 관련이 있습니다[2][3].

########################################################################
# 피싱 메일을 어떻게 구별하나요?
########################################################################

[fixed-200]  1위 점수 0.534  출처: 피싱, 피싱, 피싱
피싱 메일은 신뢰할 수 있는 메일 주소를 가장하여 무작위로 발송됩니다. 예를 들어, 정상적인 메일 주소처럼 보이지만 실제로는 사기꾼이 보낸 경우가 많습니다. 또한 신용카드 번호나 비밀번호 등 중요한 정보를 입력하도록 요구하는 점이 특징입니다. 첨부파일이 없고 단순한 HTML 메일 형태로 URL을 숨기는 경우

In [27]:
question = "SQL 삽입 공격을 막으려면 어떻게 해야 하나요?"

for name in ["fixed-200", "paragraphs-200"]:
    text, hits = answer(question, name, k=3)
    print(f"[{name}] 1위 점수 {hits[0]['score']:.3f}")
    print(text)
    print()

[fixed-200] 1위 점수 0.675
제공된 자료에서 SQL 삽입 공격을 막는 구체적인 방법은 찾을 수 없습니다. 다만, SQL 삽입은 응용 프로그램의 보안 허점을 이용해 악의적인 SQL문을 실행시키는 공격이므로, 입력값 검증과 쿼리문 작성 시 주의가 필요할 것으로 추정됩니다. 하지만 자세한 방어 방법은 자료에 포함되어 있지 않습니다. [1][2][3]

[paragraphs-200] 1위 점수 0.658
제공된 자료에서는 SQL 삽입 공격을 막는 구체적인 방법에 대해 언급하고 있지 않습니다. 따라서 "SQL 삽입 공격을 막으려면 어떻게 해야 하나요?"에 대한 답변은 제공된 자료에서 찾을 수 없습니다.



In [28]:
print("「SQL 삽입」 문서의 절 제목:")
for title in re.findall(r"^==+ .+ ==+$", docs["SQL 삽입"], re.M):
    print("  ", title)

「SQL 삽입」 문서의 절 제목:
   == 예 ==
   == Blind SQL 삽입 ==
   == 방어 ==
   === 준비된 선언 ===
   === 이스케이프 ===
   == 같이 보기 ==
   == 각주 ==
   == 외부 링크 ==


In [29]:
hits = search_db(question, "paragraphs-200", k=10)

for rank, hit in enumerate(hits, 1):
    mark = "   ← 방어 방법이 여기 있다!" if "준비된 선언" in hit["text"] else ""
    head = " ".join(hit["text"].split())[:45]
    print(f"{rank:2}위 [{hit['score']:.3f}] ({hit['source']:9s}) {head}{mark}")

 1위 [0.658] (SQL 삽입   ) SQL 삽입(영어: SQL Injection, SQL 인젝션, SQL 주입)은 응
 2위 [0.540] (SQL 삽입   ) Blind SQL 삽입은 평범한 SQL 삽입과 같이 원하는 데이터를 가져올 쿼리를
 3위 [0.523] (SQL 삽입   ) 하지만 공격자가 만약 유저네임에 admin, 패스워드에 password' OR 1
 4위 [0.432] (정보 보안    ) DB 또는 어플리케이션: 외부자, 내부자가 DB내에 저장된 정보에 불법접근하는 것
 5위 [0.405] (SQL 삽입   ) 안전한 입출력 처리 == 각주 == == 외부 링크 == Complete Refe
 6위 [0.399] (SQL 삽입   ) == 방어 == === 준비된 선언 === PHP 5.2부터 사용할 수 있다. 준   ← 방어 방법이 여기 있다!
 7위 [0.390] (SQL 삽입   ) DBMS가 쿼리문을 분석하고 최적화를 진행한다. 실행: 플레이스홀더에 입력할 사용   ← 방어 방법이 여기 있다!
 8위 [0.366] (사이트 간 스크립팅) 공격자는 해당 사이트에 가입하고, 사이트에서 볼 수 있는 사람들의 실제 이름을 알
 9위 [0.364] (피싱       ) NIST(미국 국립표준기술연구소)는 피싱 공격으로 비밀번호가 유출되더라도 계정 접
10위 [0.315] (SQL 삽입   ) Defacing website with SQL injection by sploit


In [30]:
text, hits = answer("준비된 선언으로 SQL 삽입을 막는 방법", "paragraphs-200", k=3)
print(text)

준비된 선언은 SQL 쿼리문과 사용자 입력을 분리하여 SQL 삽입 공격을 방어하는 방법입니다. 먼저 쿼리문을 데이터베이스에 전송할 때 사용자 입력 대신 플레이스홀더를 사용하고, 이후에 사용자 입력 값을 별도로 바인딩하여 실행합니다. 이렇게 하면 악의적인 SQL 코드가 쿼리문에 삽입되는 것을 방지할 수 있습니다. PHP 5.2부터 지원되며, 효과적인 SQL 삽입 방어 기법입니다[2].
